In [1]:
from qiskit_ibm_runtime import QiskitRuntimeService

# One-time setup: save each account with a distinct name
QiskitRuntimeService.save_account(
    channel="ibm_quantum_platform",
    token="",
    instance="",
    name="account2",
    overwrite=True,
)



In [2]:
# ---------------------------------------------------------------------
# Setup
# ---------------------------------------------------------------------
!pip install qiskit qiskit-ibm-runtime scipy --quiet

import json
import numpy as np
from scipy.optimize import minimize

from qiskit.circuit.library import efficient_su2
from qiskit.quantum_info import SparsePauliOp
from qiskit.transpiler.preset_passmanagers import generate_preset_pass_manager

from qiskit_ibm_runtime import QiskitRuntimeService, EstimatorV2 as Estimator


[notice] A new release of pip is available: 26.1.2 -> 26.2
[notice] To update, run: pip install --upgrade pip


In [6]:
# 1. The Hamiltonian (H2 molecule, 2 qubits)
hamiltonian = SparsePauliOp.from_list([
    ("II", -1.052373245),
    ("IZ",  0.397936080),
    ("ZI", -0.397936080),
    ("ZZ", -0.011280104),
    ("XX",  0.180931360),
])


In [4]:
# ---------------------------------------------------------------------
# 1. Authenticate + pick a backend
# ---------------------------------------------------------------------
# First time only: QiskitRuntimeService.save_account(channel="ibm_quantum", token="YOUR_API_TOKEN")
service = QiskitRuntimeService()

backend = service.least_busy(operational=True, simulator=False, min_num_qubits=2)
print("Using backend:", backend.name)

Using backend: ibm_fez


In [8]:
# ---------------------------------------------------------------------
# 3. Ansatz — transpiled once for the target backend's native gate set
# ---------------------------------------------------------------------
ansatz = efficient_su2(num_qubits=2, reps=2, entanglement="linear")
print("Number of parameters:", ansatz.num_parameters)

pm = generate_preset_pass_manager(target=backend.target, optimization_level=1)
isa_ansatz = pm.run(ansatz)
isa_hamiltonian = hamiltonian.apply_layout(isa_ansatz.layout)



Number of parameters: 12


In [10]:
# ---------------------------------------------------------------------
# 4. Cost function — ONE job per iteration, with job-ID + theta logging
#    for crash recovery (same idea as the qBraid version).
# ---------------------------------------------------------------------
estimator = Estimator(mode=backend)

job_log = []  # list of (theta_list, job_id) tuples

def save_job_log(path="ibm_job_log.json"):
    with open(path, "w") as f:
        json.dump(job_log, f)

def cost_function(theta):
    job = estimator.run([(isa_ansatz, isa_hamiltonian, [theta])])

    job_log.append((theta.tolist(), job.job_id()))
    save_job_log()  # persist immediately, survives a kernel crash

    result = job.result()
    energy = result[0].data.evs[0]
    print(f"  energy = {energy:.6f}   job = {job.job_id()}")
    return energy

In [11]:
# ---------------------------------------------------------------------
# 5. Classical optimization loop (COBYLA)
# ---------------------------------------------------------------------
np.random.seed(42)
x0 = np.random.uniform(-np.pi, np.pi, ansatz.num_parameters)

print("\nStarting VQE optimization on", backend.name, "...")
result = minimize(
    cost_function,
    x0,
    method="COBYLA",
    options={"maxiter": 30, "tol": 1e-3},
)

print("\n=== VQE RESULT ===")
print("Estimated ground-state energy:", result.fun, "Hartree")
print("Reference (exact) H2 ground energy: ~ -1.137 Hartree")
print("Optimal parameters:", result.x)


Starting VQE optimization on ibm_fez ...
  energy = -0.935338   job = d9nngkgqs0bc73e3l4s0
  energy = -1.257082   job = d9nnkkoqs0bc73e3l9m0
  energy = -1.336064   job = d9nnkumij12s73fu3ueg
  energy = -1.276636   job = d9nnlmuij12s73fu3v6g
  energy = -1.193835   job = d9nnlrmij12s73fu3vbg
  energy = -1.046937   job = d9nnm0ksfqic73ar8nv0
  energy = -1.409793   job = d9nnm5oqs0bc73e3lb50
  energy = -1.466929   job = d9nnmaksfqic73ar8o8g
  energy = -1.543843   job = d9nnmfk60llc73caals0
  energy = -1.317978   job = d9nnmkksfqic73ar8oi0
  energy = -1.292673   job = d9nnmpeij12s73fu4090


RuntimeJobFailureError: 'Unable to retrieve job result. Error code 8060; Failed to execute program: A control instrument error occurred. -- https://ibm.biz/error_codes#8060'

In [12]:
# ---------------------------------------------------------------------
# 6. RECOVERY — if the kernel dies mid-run, reattach by job ID.
#    This is IBM's native pattern: service.job(job_id).
# ---------------------------------------------------------------------
with open("ibm_job_log.json") as f:
    pairs = json.load(f)  # [(theta, job_id), ...]

print(f"Loaded {len(pairs)} logged iterations")

energies = []
for i, (theta, job_id) in enumerate(pairs):
    job = service.job(job_id)
    status = str(job.status())
    if "DONE" not in status.upper():
        print(f"iteration {i}: not done yet ({status}), skipping")
        continue
    energy = job.result()[0].data.evs[0]
    energies.append((theta, energy))
    print(f"iteration {i}: energy = {energy:.6f}  (job {job_id})")

if energies:
    best_theta, best_energy = min(energies, key=lambda pair: pair[1])
    print("\nBest recovered energy:", best_energy, "Hartree")
    print("Best theta:", best_theta)
    print("Reference (exact) H2 ground energy: ~ -1.137 Hartree")

Loaded 12 logged iterations
iteration 0: energy = -0.935338  (job d9nngkgqs0bc73e3l4s0)
iteration 1: energy = -1.257082  (job d9nnkkoqs0bc73e3l9m0)
iteration 2: energy = -1.336064  (job d9nnkumij12s73fu3ueg)
iteration 3: energy = -1.276636  (job d9nnlmuij12s73fu3v6g)
iteration 4: energy = -1.193835  (job d9nnlrmij12s73fu3vbg)
iteration 5: energy = -1.046937  (job d9nnm0ksfqic73ar8nv0)
iteration 6: energy = -1.409793  (job d9nnm5oqs0bc73e3lb50)
iteration 7: energy = -1.466929  (job d9nnmaksfqic73ar8o8g)
iteration 8: energy = -1.543843  (job d9nnmfk60llc73caals0)
iteration 9: energy = -1.317978  (job d9nnmkksfqic73ar8oi0)
iteration 10: energy = -1.292673  (job d9nnmpeij12s73fu4090)
iteration 11: not done yet (ERROR), skipping

Best recovered energy: -1.5438432073481962 Hartree
Best theta: [0.21171231810125102, 3.8319215077704234, 1.4576609265440963, 0.6198895383354297, -2.1612986243157413, -1.1614501754128375, -1.776642555026645, 3.3007525789727232, 0.6353243561202211, 1.3073585636350948

In [13]:
import numpy as np
from scipy.optimize import minimize

best_theta = np.array([
    0.21171231810125102, 3.8319215077704234, 1.4576609265440963,
    0.6198895383354297, -2.1612986243157413, -1.1614501754128375,
    -1.776642555026645, 3.3007525789727232, 0.6353243561202211,
    1.3073585636350948, -3.012256461474685, 2.952530678803174
])

print("\nResuming VQE optimization from best recovered theta on", backend.name, "...")
result = minimize(
    cost_function,          # use the retry-hardened version from before
    best_theta,              # warm start, not random x0
    method="COBYLA",
    options={"maxiter": 20, "tol": 1e-3},   # fewer iterations since we're starting close
)

print("\n=== VQE RESULT (warm-started) ===")
print("Estimated ground-state energy:", result.fun, "Hartree")
print("Reference (exact) H2 ground energy: ~ -1.137 Hartree")
print("Optimal parameters:", result.x)


Resuming VQE optimization from best recovered theta on ibm_fez ...
  energy = -1.541864   job = d9nnpo4sfqic73ar8rg0
  energy = -1.242106   job = d9nnpteij12s73fu435g
  energy = -1.301643   job = d9nnq2460llc73caap4g
  energy = -1.499114   job = d9nnq7csfqic73ar8s00
  energy = -1.548180   job = d9nnqdk60llc73caapgg
  energy = -1.258557   job = d9nnqis60llc73caapm0
  energy = -1.440024   job = d9nnqq460llc73caapt0
  energy = -1.724987   job = d9nnqv4sfqic73ar8sr0
  energy = -1.614088   job = d9nnr6uij12s73fu44f0
  energy = -1.549731   job = d9nnrnc60llc73caaqu0
  energy = -1.500331   job = d9nns18qs0bc73e3lgpg
  energy = -1.657498   job = d9nns9460llc73caarjg
  energy = -1.716203   job = d9nnsl4sfqic73ar8uig
  energy = -1.756950   job = d9nnsrmij12s73fu465g
  energy = -1.595400   job = d9nnt0csfqic73ar8uvg
  energy = -1.648900   job = d9nnt5c60llc73caashg
  energy = -1.634239   job = d9nntac60llc73caasng
  energy = -1.724696   job = d9nntfmij12s73fu46v0
  energy = -1.655782   job = d9n